<a href="https://colab.research.google.com/github/keswong/phd_listing_repo/blob/main/3_12_3_Comparing_Interventions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [24]:
%reset -f
import pandas as pd
import ast
import re
from sklearn.preprocessing import MinMaxScaler
from statsmodels.stats.contingency_tables import mcnemar
from statsmodels.stats.multitest import multipletests

# Read the Excel file (update path if needed)
df = pd.read_excel(".../Teacher_Interventions.xlsx", sheet_name="teacher")

# Convert the string representation into a Python list
df["interventions"] = df["interventions"].apply(ast.literal_eval)

# Each element is like ['text'], so flatten inner lists
df["interventions"] = df["interventions"].apply(
    lambda x: [item[0] if isinstance(item, list) else item for item in x]
)

# Explode into individual rows
df = df.explode("interventions", ignore_index=True)

# Remove any leftover brackets or punctuation if present
df["interventions"] = (
    df["interventions"]
    .str.replace(r"[\[\]]", "", regex=True)
    .str.strip()
)


coding_cols = [
    "M1","M2","M3","M4","M5","M6","M7",
    "F1","F2","F3","F4","C3","C4"
]

coding_df = pd.read_excel(".../Teacher_Interventions.xlsx", sheet_name="coding")

# If the intervention text column in coding has a different name, update here
df = df.merge(
    coding_df[["interventions"] + coding_cols],
    on="interventions",
    how="left"
)

df = df.rename(columns={
    "C3": "C1",
    "C4": "C2"
})

coding_cols = [
    "M1","M2","M3","M4","M5","M6","M7",
    "F1","F2","F3","F4","C1","C2"
]

utterance_summary = (
    df
    .groupby("utterance_number", as_index=False)[coding_cols]
    .sum()
)

scaler = MinMaxScaler(feature_range=(0, 1))

utterance_summary_norm = utterance_summary.copy()
utterance_summary_norm[coding_cols] = scaler.fit_transform(
    utterance_summary[coding_cols]
)

utterance_summary_bin = utterance_summary_norm.copy()
utterance_summary_bin[coding_cols] = (utterance_summary_norm[coding_cols] > 0).astype(int)

# Read the Excel file (update path if needed)
df_teacher_created = pd.read_excel(".../coding_teacher_automated_interventions.xlsx", sheet_name="teacher-created")
df_teacher_created = df_teacher_created.drop(columns=["C1"])
df_teacher_created = df_teacher_created.rename(columns={
    "C3": "C1",
    "C4": "C2"
})

coding_cols = [
    "M1","M2","M3","M4","M5","M6","M7",
    "F1","F2","F3","F4","C1","C2"
]
df_teacher_created[coding_cols] = df_teacher_created[coding_cols].apply(
    pd.to_numeric, errors="coerce"
).fillna(0)

df = pd.read_excel(".../coding_teacher_automated_interventions.xlsx", sheet_name="automated_intervention")
df_with_PA = df[df["condition"] == "with_pedagogy"].copy()
df_with_PA = df_with_PA.drop(columns=["condition","PRS1","PRS2","PRS3",
    "PRT4","PRT5","PRT6","PRT7","PRT_M1","PRT_M2","PRT_M4","PRT_M5"])
df_with_PA = df_with_PA.rename(columns={
    "C3": "C1",
    "C4": "C2"
})

coding_cols = [
    "M1","M2","M3","M4","M5","M6","M7",
    "F1","F2","F3","F4","C1","C2"
]
df_with_PA[coding_cols] = df_with_PA[coding_cols].apply(
    pd.to_numeric, errors="coerce"
).fillna(0)
df_with_PA[coding_cols] = df_with_PA[coding_cols].astype(int)


df_without_PA = df[df["condition"] == "without_pedagogy"].copy()
df_without_PA = df_without_PA.drop(columns=["condition","PRS1","PRS2","PRS3",
    "PRT4","PRT5","PRT6","PRT7","PRT_M1","PRT_M2","PRT_M4","PRT_M5"])
df_without_PA = df_without_PA.rename(columns={
    "C3": "C1",
    "C4": "C2"
})

coding_cols = [
    "M1","M2","M3","M4","M5","M6","M7",
    "F1","F2","F3","F4","C1","C2"
]
df_without_PA[coding_cols] = df_without_PA[coding_cols].apply(
    pd.to_numeric, errors="coerce"
).fillna(0)
df_without_PA[coding_cols] = df_without_PA[coding_cols].astype(int)

# Add condition labels (optional but explicit)
utterance_summary_bin['condition'] = 'teacher'
utterance_summary_bin = utterance_summary_bin.rename(
    columns={"utterance_number": "utterance_id"}
)

# Merge on utterance_id (PAIRING happens here) - WITH_PA vs TEACHER
df_paired_without = utterance_summary_bin.merge(
    df_without_PA,
    on='utterance_id',
    suffixes=('_teacher', '_without_PA')
)

results_without = []

for col in coding_cols:
    col_teacher = f"{col}_teacher"
    col_with = f"{col}_without_PA"

    # Build 2x2 contingency table
    table = pd.crosstab(
        df_paired_without[col_teacher],
        df_paired_without[col_with]
    )

    # Ensure full 2x2 structure
    table = table.reindex(index=[0,1], columns=[0,1], fill_value=0)

    # McNemar test (exact recommended for small N)
    res = mcnemar(table, exact=True)

    results_without.append({
        "code": col,
        "teacher=1,without=0": table.loc[1,0],
        "teacher=0,without=1": table.loc[0,1],
        "teacher=0,without=0": table.loc[0,0],
        "teacher=1,without=1": table.loc[1,1],
        "p_value": res.pvalue
    })

# Apply Holm correction for Without_PA vs TEACHER
p_values_without = [res['p_value'] for res in results_without]
rejected_without, p_corrected_without, _, _ = multipletests(p_values_without, alpha=0.05, method='holm')

# Add corrected p-values to results
for i, res in enumerate(results_without):
    res['p_value_original'] = res['p_value']
    res['p_value_holm'] = p_corrected_without[i]
    res['significant_holm'] = rejected_without[i]

# Create formatted table
results_df_without = pd.DataFrame(results_without)
results_df_without['p_value_original'] = results_df_without['p_value_original'].apply(lambda x: f"{x:.4e}")
results_df_without['p_value_holm'] = results_df_without['p_value_holm'].apply(lambda x: f"{x:.4e}")
results_df_without['significant'] = results_df_without['significant_holm'].map({True: 'Yes', False: 'No'})

print("="*80)
print("Without_PA vs TEACHER - McNemar Test Results with Holm Correction")
print("="*80)
print("\nContingency table interpretation:")
print("- 'teacher=1,without=0': Teacher has code, Without_PA does not")
print("- 'teacher=0,without=1': Teacher does not have code, Without_PA does")
print("- 'teacher=0,without=0': Neither has code")
print("- 'teacher=1,without=1': Both have code")
print("-"*80)

# Display clean table
display_cols = ['code', 'teacher=1,without=0', 'teacher=0,without=1',
                'teacher=0,without=0', 'teacher=1,without=1',
                'p_value_original', 'p_value_holm']
print(results_df_without[display_cols].to_string(index=False))

Without_PA vs TEACHER - McNemar Test Results with Holm Correction

Contingency table interpretation:
- 'teacher=1,without=0': Teacher has code, Without_PA does not
- 'teacher=0,without=1': Teacher does not have code, Without_PA does
- 'teacher=0,without=0': Neither has code
- 'teacher=1,without=1': Both have code
--------------------------------------------------------------------------------
code  teacher=1,without=0  teacher=0,without=1  teacher=0,without=0  teacher=1,without=1 p_value_original p_value_holm
  M1                    9                    6                    3                   13       6.0724e-01   1.0000e+00
  M2                   29                    0                    2                    0       3.7253e-09   4.4703e-08
  M3                   15                    4                    8                    4       1.9211e-02   1.3448e-01
  M4                   13                    3                    5                   10       2.1271e-02   1.3448e-01
  M5     

In [21]:
%reset -f
import pandas as pd
import ast
import re
from sklearn.preprocessing import MinMaxScaler
from statsmodels.stats.contingency_tables import mcnemar
from statsmodels.stats.multitest import multipletests

# Read the Excel file (update path if needed)
df = pd.read_excel(".../Teacher_Interventions.xlsx", sheet_name="teacher")

# Convert the string representation into a Python list
df["interventions"] = df["interventions"].apply(ast.literal_eval)

# Each element is like ['text'], so flatten inner lists
df["interventions"] = df["interventions"].apply(
    lambda x: [item[0] if isinstance(item, list) else item for item in x]
)

# Explode into individual rows
df = df.explode("interventions", ignore_index=True)

# Remove any leftover brackets or punctuation if present
df["interventions"] = (
    df["interventions"]
    .str.replace(r"[\[\]]", "", regex=True)
    .str.strip()
)


coding_cols = [
    "M1","M2","M3","M4","M5","M6","M7",
    "F1","F2","F3","F4","C3","C4"
]

coding_df = pd.read_excel(".../Teacher_Interventions.xlsx", sheet_name="coding")

# If the intervention text column in coding has a different name, update here
df = df.merge(
    coding_df[["interventions"] + coding_cols],
    on="interventions",
    how="left"
)

df = df.rename(columns={
    "C3": "C1",
    "C4": "C2"
})

coding_cols = [
    "M1","M2","M3","M4","M5","M6","M7",
    "F1","F2","F3","F4","C1","C2"
]

utterance_summary = (
    df
    .groupby("utterance_number", as_index=False)[coding_cols]
    .sum()
)

scaler = MinMaxScaler(feature_range=(0, 1))

utterance_summary_norm = utterance_summary.copy()
utterance_summary_norm[coding_cols] = scaler.fit_transform(
    utterance_summary[coding_cols]
)

utterance_summary_bin = utterance_summary_norm.copy()
utterance_summary_bin[coding_cols] = (utterance_summary_norm[coding_cols] > 0).astype(int)

# Read the Excel file (update path if needed)
df_teacher_created = pd.read_excel(".../coding_teacher_automated_interventions.xlsx", sheet_name="teacher-created")
df_teacher_created = df_teacher_created.drop(columns=["C1"])
df_teacher_created = df_teacher_created.rename(columns={
    "C3": "C1",
    "C4": "C2"
})

coding_cols = [
    "M1","M2","M3","M4","M5","M6","M7",
    "F1","F2","F3","F4","C1","C2"
]
df_teacher_created[coding_cols] = df_teacher_created[coding_cols].apply(
    pd.to_numeric, errors="coerce"
).fillna(0)

df = pd.read_excel(".../coding_teacher_automated_interventions.xlsx", sheet_name="automated_intervention")
df_with_PA = df[df["condition"] == "with_pedagogy"].copy()
df_with_PA = df_with_PA.drop(columns=["condition","PRS1","PRS2","PRS3",
    "PRT4","PRT5","PRT6","PRT7","PRT_M1","PRT_M2","PRT_M4","PRT_M5"])
df_with_PA = df_with_PA.rename(columns={
    "C3": "C1",
    "C4": "C2"
})

coding_cols = [
    "M1","M2","M3","M4","M5","M6","M7",
    "F1","F2","F3","F4","C1","C2"
]
df_with_PA[coding_cols] = df_with_PA[coding_cols].apply(
    pd.to_numeric, errors="coerce"
).fillna(0)
df_with_PA[coding_cols] = df_with_PA[coding_cols].astype(int)


df_without_PA = df[df["condition"] == "without_pedagogy"].copy()
df_without_PA = df_without_PA.drop(columns=["condition","PRS1","PRS2","PRS3",
    "PRT4","PRT5","PRT6","PRT7","PRT_M1","PRT_M2","PRT_M4","PRT_M5"])
df_without_PA = df_without_PA.rename(columns={
    "C3": "C1",
    "C4": "C2"
})

coding_cols = [
    "M1","M2","M3","M4","M5","M6","M7",
    "F1","F2","F3","F4","C1","C2"
]
df_without_PA[coding_cols] = df_without_PA[coding_cols].apply(
    pd.to_numeric, errors="coerce"
).fillna(0)
df_without_PA[coding_cols] = df_without_PA[coding_cols].astype(int)

# Add condition labels (optional but explicit)
utterance_summary_bin['condition'] = 'teacher'
utterance_summary_bin = utterance_summary_bin.rename(
    columns={"utterance_number": "utterance_id"}
)

# Merge on utterance_id (PAIRING happens here) - WITH_PA vs TEACHER
df_paired_with = utterance_summary_bin.merge(
    df_with_PA,
    on='utterance_id',
    suffixes=('_teacher', '_with_PA')
)

results_with = []

for col in coding_cols:
    col_teacher = f"{col}_teacher"
    col_with = f"{col}_with_PA"

    # Build 2x2 contingency table
    table = pd.crosstab(
        df_paired_with[col_teacher],
        df_paired_with[col_with]
    )

    # Ensure full 2x2 structure
    table = table.reindex(index=[0,1], columns=[0,1], fill_value=0)

    # McNemar test (exact recommended for small N)
    res = mcnemar(table, exact=True)

    results_with.append({
        "code": col,
        "teacher=1,with=0": table.loc[1,0],
        "teacher=0,with=1": table.loc[0,1],
        "teacher=0,with=0": table.loc[0,0],
        "teacher=1,with=1": table.loc[1,1],
        "p_value": res.pvalue
    })

# Apply Holm correction for WITH_PA vs TEACHER
p_values_with = [res['p_value'] for res in results_with]
rejected_with, p_corrected_with, _, _ = multipletests(p_values_with, alpha=0.05, method='holm')

# Add corrected p-values to results
for i, res in enumerate(results_with):
    res['p_value_original'] = res['p_value']
    res['p_value_holm'] = p_corrected_with[i]
    res['significant_holm'] = rejected_with[i]

# Create formatted table
results_df_with = pd.DataFrame(results_with)
results_df_with['p_value_original'] = results_df_with['p_value_original'].apply(lambda x: f"{x:.4e}")
results_df_with['p_value_holm'] = results_df_with['p_value_holm'].apply(lambda x: f"{x:.4e}")
results_df_with['significant'] = results_df_with['significant_holm'].map({True: 'Yes', False: 'No'})

print("="*80)
print("WITH_PA vs TEACHER - McNemar Test Results with Holm Correction")
print("="*80)
print("\nContingency table interpretation:")
print("- 'teacher=1,with=0': Teacher has code, With_PA does not")
print("- 'teacher=0,with=1': Teacher does not have code, With_PA does")
print("- 'teacher=0,with=0': Neither has code")
print("- 'teacher=1,with=1': Both have code")
print("-"*80)

# Display clean table
display_cols = ['code', 'teacher=1,with=0', 'teacher=0,with=1',
                'teacher=0,with=0', 'teacher=1,with=1',
                'p_value_original', 'p_value_holm']
print(results_df_with[display_cols].to_string(index=False))

WITH_PA vs TEACHER - McNemar Test Results with Holm Correction

Contingency table interpretation:
- 'teacher=1,with=0': Teacher has code, With_PA does not
- 'teacher=0,with=1': Teacher does not have code, With_PA does
- 'teacher=0,with=0': Neither has code
- 'teacher=1,with=1': Both have code
--------------------------------------------------------------------------------
code  teacher=1,with=0  teacher=0,with=1  teacher=0,with=0  teacher=1,with=1 p_value_original p_value_holm
  M1                 0                 9                 0                22       3.9062e-03   2.7344e-02
  M2                29                 0                 2                 0       3.7253e-09   4.8429e-08
  M3                 9                 7                 5                10       8.0362e-01   1.0000e+00
  M4                23                 0                 8                 0       2.3842e-07   2.3842e-06
  M5                 1                15                15                 0       5.1880e